# Generating the Dataset

For this example, we will work in low dimension
for succinctness.
The following code snippet generates 1000 examples
with 2-dimensional features drawn 
from a standard normal distribution.
The resulting design matrix $\mathbf{X}$
belongs to $\mathbb{R}^{1000 \times 2}$. 
We generate each label by applying 
a *ground truth* linear function, 
corrupting them via additive noise $\boldsymbol{\epsilon}$, 
drawn independently and identically for each example:

**$$\mathbf{y}= \mathbf{X} \mathbf{w} + b + \boldsymbol{\epsilon}.$$**

For convenience we assume that $\boldsymbol{\epsilon}$ is drawn 
from a normal distribution with mean $\mu= 0$ 
and standard deviation $\sigma = 0.01$.
Note that for object-oriented design
we add the code to the `__init__` method of a subclass of `d2l.DataModule` (introduced in :numref:`oo-design-data`). 
It is good practice to allow the setting of any additional hyperparameters. 
We accomplish this with `save_hyperparameters()`. 
The `batch_size` will be determined later.


## Generating the Dataset


In [122]:
%matplotlib inline
import random
import torch
from d2l import torch as d2l

In [123]:
class SyntheticRegressionData(d2l.DataModule):  #@save
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000,
                 batch_size=32):
        super().__init__()
        self.save_hyperparameters()
        n = num_train + num_val
        self.X = torch.randn(n, len(w))
        noise = torch.randn(n, 1) * noise
        self.y = torch.matmul(self.X, w.reshape((-1, 1))) + b + noise

In [124]:
data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)

In [125]:
print('features:', data.X[0],'\nlabel:', data.y[0])

features: tensor([-1.2925, -0.0861]) 
label: tensor([1.9109])


In [126]:
print('features:', data.X,'\nlabel:', data.y)

features: tensor([[-1.2925, -0.0861],
        [ 0.6649, -1.6343],
        [ 0.6614,  1.6368],
        ...,
        [-0.3181, -0.7131],
        [-0.9589, -0.3635],
        [-0.2583,  0.9241]]) 
label: tensor([[ 1.9109],
        [11.0653],
        [-0.0537],
        ...,
        [ 5.9973],
        [ 3.5237],
        [ 0.5349]])


## Reading the Dataset


In [127]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    if train:
        indices = list(range(0, self.num_train))
        # The examples are read in random order
        random.shuffle(indices)
    else:
        indices = list(range(self.num_train, self.num_train+self.num_val))
    for i in range(0, len(indices), self.batch_size):

        # Batches the index per batch_size (32 here)
        # If not shuffled for the first iteration
        # batches_indices == tensor([0, 1, 2 ... 30, 31])
        batch_indices = torch.tensor(indices[i: i+self.batch_size])
        # print(batch_indices)

        # 'yield' is like a partial-return that continues the execution when we use next()
        yield self.X[batch_indices], self.y[batch_indices]



In [128]:
# Equiv to X, y = next(iter(data.train_dataloader())) if we don't the next iterations
loader = iter(data.train_dataloader())
X, y = next(loader)
# To iterate to the next batch
# A, b = next(loader)


In [129]:
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


## Concise Implementation of the Data Loader

In [130]:
@d2l.add_to_class(d2l.DataModule)  #@save
def get_tensorloader(self, tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)
    dataset = torch.utils.data.TensorDataset(*tensors)
    return torch.utils.data.DataLoader(dataset, self.batch_size,
                                       shuffle=train)

In [ ]:
@d2l.add_to_class(SyntheticRegressionData)  #@save
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)
    return self.get_tensorloader((self.X, self.y), train, i)

In [ ]:
X, y = next(iter(data.train_dataloader()))
print('X shape:', X.shape, '\ny shape:', y.shape)

X shape: torch.Size([32, 2]) 
y shape: torch.Size([32, 1])


`len` Returns the numbers of minibatches

In [ ]:
len(data.train_dataloader())
# len(data.val_dataloader())

32


## Exercises

1. What will happen if the number of examples cannot be divided by the batch size. How would you change this behavior by specifying a different argument by using the framework's API?
1. Suppose that we want to generate a huge dataset, where both the size of the parameter vector `w` and the number of examples `num_examples` are large.
    1. What happens if we cannot hold all data in memory?
    1. How would you shuffle the data if it is held on disk? Your task is to design an *efficient* algorithm that does not require too many random reads or writes. Hint: [pseudorandom permutation generators](https://en.wikipedia.org/wiki/Pseudorandom_permutation) allow you to design a reshuffle without the need to store the permutation table explicitly :cite:`Naor.Reingold.1999`. 
1. Implement a data generator that produces new data on the fly, every time the iterator is called. 
1. How would you design a random data generator that generates *the same* data each time it is called?


1. 

1.1. By default, the last batch contains the rest of the examples.

Here, $1000 = 31 \times 32 + 8$. 

The first 31 batches contain 32 examples each.

The 32th batch contain 8 examples. 

1.2. To remove the incomplete batch, we specify `drop_last = True`:  

```python
DataLoader(
    dataset,
    batch_size=self.batch_size,
    shuffle=train,
    drop_last=True
)
```

2. A.

If we cannot hold all the data in memory it'll either: 
- Raise OutOfMemory Error
- Use Disk Swap (Slow)


The solution is to load data per batch 

In [ ]:
def data_generator(w, b, batch_size, num_batches):
    for _ in range(num_batches):
        X = torch.randn(batch_size, len(w))
        y = X @ w + b
        yield X, y

2. B.

We use the `Pseudorandom Permutation (PRP)` 

We First divide for example `1M` blocks into `1k` blocks containing `1k` examples each
- We shuffle the blocks indices and read 

For each epoch (One complete pass through the training set), we should:

- Choose a random seed $s$
- Use a pseudorandom permutation $P_s$ to permute block numbers
- Read Block with one contigious disk read
- produce its minibatches
- Continue with the other blocks

Example Code:

In [138]:
def pseudorandom_permutation():
    raise NotImplemented
def read_contiguous_block_from_disk():
    raise NotImplemented
def split_into_batches():
    raise NotImplemented

def disk_batches(num_blocks, seed):
    for position in range(num_blocks):
        block_id = pseudorandom_permutation(position, seed)

        block = read_contiguous_block_from_disk(block_id)
        random.Random(seed + block_id).shuffle(block)

        for batch in split_into_batches(block):
            yield batch

Generally, we choose the largest block—or group of blocks—that comfortably fits within the available RAM buffer:

3.
We can generate per example but we generally want a batch so we adapt for that 

One sized batch example:

In [ ]:
def data_generator(w, b, noise_std, num_examples):
    for _ in range(num_examples):
        x = torch.randn(1, len(w))
        noise = torch.randn(1) * noise_std
        y = x @ w + b + noise

        yield x, y

General example:

In [145]:
def data_generator(w, b, noise_std, batch_size, num_batches):
    for _ in range(num_batches):
        X = torch.randn(batch_size, len(w))
        noise = torch.randn(batch_size) * noise_std
        y = X @ w + b + noise

        yield X, y

4. 
We use a fixed random seed: 

In [ ]:
def reproducible_data_generator(
    w, b, noise_std, batch_size, num_batches, seed=42
):
    rng = torch.Generator()
    rng.manual_seed(seed)

    for _ in range(num_batches):
        X = torch.randn(
            batch_size,
            len(w),
            generator=rng
        )

        noise = torch.randn(
            batch_size,
            1,
            generator=rng
        ) * noise_std

        y = X @ w.reshape(-1, 1) + b + noise

        yield X, y